# LLM/VLM Inference Benchmark

This notebook compares inference backends (Transformers, vLLM, llama.cpp, and local Qwen agent).
Each backend cell records model load time and per-prompt latency for a shared prompt set (some with images).

In [1]:
from __future__ import annotations

import time
from pathlib import Path
from typing import Any

import pandas as pd


In [5]:
# --- Configuration
MODEL_ID = "./modeles/Qwen3.5-27B"  # update if needed
DEVICE = "cuda"
IMAGE_DIR = Path('.')

# Backend-specific settings (set paths if you use alternate VLM builds)
VLLM_MODEL_ID = MODEL_ID
VLLM_ENABLE_IMAGES = True
VLLM_MAX_TOKENS = 64

LLAMACPP_GGUF_PATH = None  # set to a local GGUF VLM path for llama.cpp
LLAMACPP_MM_PROJ_PATH = None  # set to a mmproj path for images (LLaVA-style)
LLAMACPP_CHAT_FORMAT = "llava-1-5"
LLAMACPP_MAX_TOKENS = 64

PROMPTS = [
    {"id": "txt-1", "text": "Resume the following in one sentence: Transformers are widely used in NLP."},
    {"id": "txt-2", "text": "List three safety checks before deploying an LLM."},
    {"id": "img-1", "text": "Describe what you see.", "image": IMAGE_DIR / "rue.png"},
    {"id": "img-2", "text": "Describe the scene in detail.", "image": IMAGE_DIR / "table_ronde.jpg"},
    {"id": "img-3", "text": "What object is shown and what is it used for?", "image": IMAGE_DIR / "missile.png"},
]


In [6]:
# --- Utilities
def _now() -> float:
    return time.perf_counter()

def _exists_image(path: Path | None) -> bool:
    return path is not None and path.exists()

def run_benchmark(
    backend: str,
    load_fn,
    infer_fn,
    prompts: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    t0 = _now()
    print("chargement du modele ...")
    ctx = load_fn()
    print("fin du loading")
    load_time = _now() - t0

    print("démarrage de l'inférence ...")

    for i , item in enumerate(prompts):
        t1 = _now()
        infer_fn(ctx, item)
        latency = _now() - t1
        results.append({
            "backend": backend,
            "prompt_id": item["id"],
            "has_image": _exists_image(item.get("image")),
            "load_time_s": load_time,
            "latency_s": latency,
        })
        print("fin de l'inférence ...",i+1)

    return results


In [7]:
# --- Results storage
all_results: list[dict[str, Any]] = []


In [8]:
# === Backend 1: Hugging Face Transformers (text-only, image if supported)
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor
    from PIL import Image

    def _hf_load():
        tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        try:
            proc = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        except Exception:
            proc = None
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto" if DEVICE == "cuda" else None,
            trust_remote_code=True
        )
        return {"model": model, "tok": tok, "proc": proc}

    def _hf_infer(ctx, item):
        model = ctx["model"]
        tok = ctx["tok"]
        proc = ctx["proc"]
        text = item["text"]
        image_path = item.get("image")
        if proc is not None and _exists_image(image_path):
            image = Image.open(image_path).convert("RGB")
            inputs = proc(text=text, images=image, return_tensors="pt")
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
        else:
            inputs = tok(text, return_tensors="pt").to(model.device)
        _ = model.generate(**inputs, max_new_tokens=64)

    all_results.extend(run_benchmark("transformers", _hf_load, _hf_infer, PROMPTS))
except Exception as e:
    print("Transformers backend skipped:", e)


chargement du modele ...


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
[transformers] Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
Loading weights: 100%|██████████| 851/851 [00:00<00:00, 3622.27it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


fin du loading
démarrage de l'inférence ...
Transformers backend skipped: probability tensor contains either `inf`, `nan` or element < 0


In [7]:
try:
    from vllm import LLM, SamplingParams

    def _vllm_load():
        llm = LLM(model=VLLM_MODEL_ID)
        params = SamplingParams(max_tokens=VLLM_MAX_TOKENS)
        supports_images = VLLM_ENABLE_IMAGES
        return {"llm": llm, "params": params, "supports_images": supports_images}

    def _vllm_infer(ctx, item):
        text = item["text"]
        image_path = item.get("image")
        if ctx["supports_images"] and _exists_image(image_path):
            try:
                request = {
                    "prompt": text,
                    "multi_modal_data": {"image": str(image_path)},
                }
                _ = ctx["llm"].generate([request], ctx["params"])
                return
            except Exception:
                pass
        _ = ctx["llm"].generate([text], ctx["params"])

    all_results.extend(run_benchmark("vllm", _vllm_load, _vllm_infer, PROMPTS))
except Exception as e:
    print("vLLM backend skipped:", e)


chargement du modele ...
INFO 06-01 11:47:31 [utils.py:278] non-default args: {'disable_log_stats': True, 'model': './modeles/Qwen3.5-27B'}
INFO 06-01 11:47:31 [model.py:617] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 06-01 11:47:31 [model.py:1752] Using max model len 262144
INFO 06-01 11:47:31 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-01 11:47:31 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-01 11:47:31 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=20406) INFO 06-01 11:47:42 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='./modeles/Qwen3.5-27B', speculative_config=None, tokenizer='./modeles/Qwen3.5-27B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_end

(EngineCore pid=20406) Process EngineCore:
(EngineCore pid=20406) Traceback (most recent call last):
(EngineCore pid=20406)   File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=20406)     self.run()
(EngineCore pid=20406)   File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore pid=20406)     self._target(*self._args, **self._kwargs)
(EngineCore pid=20406)   File "/home/etudiant_adm/visionAssist/VisionAssist/.venv/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1169, in run_engine_core
(EngineCore pid=20406)     raise e
(EngineCore pid=20406)   File "/home/etudiant_adm/visionAssist/VisionAssist/.venv/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1139, in run_engine_core
(EngineCore pid=20406)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=20406)   File "/home/etudiant_adm/visionAssist/VisionAssist/.venv/lib/python3.10/site-packages/vllm/tracing/otel.

vLLM backend skipped: Engine core initialization failed. See root cause above. Failed core proc(s): {'EngineCore': 1}


In [11]:
import requests
import json

url = "http://0.0.0.0:8000/v1/chat/completions"
model = "./langage/modeles/Qwen3.5-27B-4bit-bitsandbytes"
def serving_avec_vllm(url:str, modele : str, prompt_systeme :str , prompt_user :str , temperature : float = 0.7,  max_tokens : int = 1024):
  payload = {
    "model": modele,
    "messages": [
      {
        "role": "system",
        "content": prompt_systeme
      },
      {
        "role": "user",
        "content": prompt_user
      }
    ],
    "temperature": temperature,
    "max_tokens": max_tokens
  }

  headers = {
      "Content-Type": "application/json"
  }

  try:
      response = requests.post(url, headers=headers, data=json.dumps(payload))
      response.raise_for_status()
      result = response.json()
      print("Réponse du modèle :")
      print(result['choices'][0]['message']['content'])

  except requests.exceptions.RequestException as e:
      print(f"Erreur lors de la requête : {e}")


serving_avec_vllm(url,model,"tu es un assistant intelligent qui répond de manière concise et précise","est ce que hamid est un zgeg")

Erreur lors de la requête : HTTPConnectionPool(host='0.0.0.0', port=8000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError("HTTPConnection(host='0.0.0.0', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))


In [ ]:
#### lancement de vllm 

INFO 06-01 11:35:02 [utils.py:278] non-default args: {'disable_log_stats': True, 'model': './modeles/Qwen3.5-27B'}
INFO 06-01 11:35:02 [model.py:617] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 06-01 11:35:02 [model.py:1752] Using max model len 262144
INFO 06-01 11:35:02 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=17475) INFO 06-01 11:35:05 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='./modeles/Qwen3.5-27B', speculative_config=None, tokenizer='./modeles/Qwen3.5-27B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None,

(EngineCore pid=17475) Process EngineCore:
(EngineCore pid=17475) Traceback (most recent call last):
(EngineCore pid=17475)   File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=17475)     self.run()
(EngineCore pid=17475)   File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore pid=17475)     self._target(*self._args, **self._kwargs)
(EngineCore pid=17475)   File "/home/etudiant_adm/visionAssist/VisionAssist/.venv/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1169, in run_engine_core
(EngineCore pid=17475)     raise e
(EngineCore pid=17475)   File "/home/etudiant_adm/visionAssist/VisionAssist/.venv/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1139, in run_engine_core
(EngineCore pid=17475)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=17475)   File "/home/etudiant_adm/visionAssist/VisionAssist/.venv/lib/python3.10/site-packages/vllm/tracing/otel.

vLLM backend skipped: Engine core initialization failed. See root cause above. Failed core proc(s): {'EngineCore': 1}


In [4]:
# === Backend 4: Local QwenAgent (text + image supported)
try:
    import uuid
    from api.schemas.broker import BrokerRequest
    from api.services.agent import QwenAgent
    from api.services.model import model_service

    def _agent_load():
        if model_service.model is None or model_service.processor is None:
            model_service.load()
        return {"agent": QwenAgent(model_service)}

    def _agent_infer(ctx, item):
        req = BrokerRequest(
            request_id=str(uuid.uuid4()),
            session_id="bench",
            text=item["text"],
            image_url=str(item["image"]) if _exists_image(item.get("image")) else None,
        )
        _ = ctx["agent"].handle(req)

    all_results.extend(run_benchmark("local-agent", _agent_load, _agent_infer, PROMPTS))
except Exception as e:
    print("Local agent backend skipped:", e)

# --- Results summary
df = pd.DataFrame(all_results)
df.sort_values(["backend", "prompt_id"]) if not df.empty else df


/home/etudiant_adm/visionAssist/VisionAssist/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


settings model_id='./langage/modeles/Qwen3.5-27B' load_in_4bit=True max_new_tokens=1024 temperature=0.7 top_p=0.8 do_sample=True enable_thinking=False host='0.0.0.0' port=8000 max_image_size_mb=10


Chargement du modèle:   0%|          | 0/2 [00:00<?, ?étape/s]

Local agent backend skipped: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './langage/modeles/Qwen3.5-27B'. Use `repo_type` argument if needed.


NameError: name 'all_results' is not defined